Train/Test Split

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

#insira aqui o caminho para os dados limpos que você quer modelar:
DADOS_LIMPOS = Path("C:/Users/Gamer GTX/sintese/projeto trainee/Projeto-Trainee-I-Dados-2026.1/dados/dados_limpos.csv")

if not DADOS_LIMPOS.exists():
    raise FileNotFoundError("dados/dados_limpos.csv")

transpondo a coluna "Class" no indice 0 do dataframe:

In [3]:
df_limpo = pd.read_csv(DADOS_LIMPOS)
transpondo_class = df_limpo.pop("Class")
df_limpo.insert(0, "Class", transpondo_class)


C:\Users\Gamer GTX\AppData\Local\Temp\ipykernel_15184\36720065.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_limpo.insert(0, "Class", transpondo_class)


Em X, temos os genes como variaveis preditoras. Em y, temos a variavel target (tipo de tumor)

Estabelecendo em 20% o volume de dados para teste e 80% para treino

In [4]:
genes = df_limpo.drop(columns=["Class"])
tumores = df_limpo["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    genes,
    tumores,
    test_size=0.2,
    random_state=42
)

Normalizando os dados

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
scaler.set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

12338560


Para realização do Label enconding verificamos inicialmente se existe alguma variável categórica no conjunto de treinamento X, e depois aplicamos o label enconding para o conjunto de dados Y, que representam o target: tipos de cancêr

In [6]:
colunas_categoricas = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Quantidade de colunas categóricas em X: {len(colunas_categoricas)}")

Quantidade de colunas categóricas em X: 0


In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

for idx, classe in enumerate(le.classes_):
    print(f"{classe} -> {idx}")

BRCA -> 0
COAD -> 1
KIRC -> 2
LUAD -> 3
PRAD -> 4


In [ ]:
import numpy as np
import shap 
from sklearn.ensemble import RandomForestClassifier

rf_inicial = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_inicial.fit(X_train_scaled, y_train)

importancias = pd.Series(rf_inicial.feature_importances_, index=X_train_scaled.columns)

features_grande = importancias[importancias > 0.005].index.to_list()

X_train_scaled = X_train_scaled[features_grande]
X_test_scaled = X_test_scaled[features_grande]

print(X_train_scaled.size)

14080


Redução do dataset através de RFC utilizando valores de importância acima de 0.5%, para evitar o descarte de variáveis que possam ser relevantes para o modelo e retirar apenas os ruídos.

In [ ]:
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_final.fit(X_train_scaled, y_train)

amostra_shap = X_train_scaled.sample(n=min(5000, len(X_train_scaled)), random_state=42)

explainer = shap.TreeExplainer(rf_final)
shap_valores = explainer(amostra_shap)
vals = shap_valores.values
if vals.ndim == 3:
    vals = vals[:,:,1]
media_shap = np.abs(vals).mean(axis=0)
importancia_shap = pd.Series(media_shap, index=X_train_scaled.columns)
features_finais = importancia_shap.sort_values(ascending=False).head(50).index.to_list()

X_train_scaled = X_train_scaled[features_finais]
X_test_scaled = X_test_scaled[features_finais]


50


Com o novo X_train utiliza-se RFC novamente para prever as variáveis úteis, depois foi utilizado o SHAP para obter as maiores importâncias das features restantes e assim reduzir o X_train_scaled para as 50 features mais relevantes

In [23]:
for i, gene in enumerate(features_finais,1):
    print(f"{i} : {gene}")

1 : gene_3921
2 : gene_12013
3 : gene_7896
4 : gene_4274
5 : gene_7965
6 : gene_6816
7 : gene_6836
8 : gene_1189
9 : gene_7238
10 : gene_2037
11 : gene_15591
12 : gene_9652
13 : gene_19608
14 : gene_6355
15 : gene_2638
16 : gene_13355
17 : gene_3440
18 : gene_7026
19 : gene_6594
20 : gene_9380
21 : gene_2747
22 : gene_6530
23 : gene_7747
24 : gene_6926
25 : gene_17801
26 : gene_2448
27 : gene_2083
28 : gene_16338
29 : gene_16133
30 : gene_11124
31 : gene_628
32 : gene_11349
33 : gene_30
34 : gene_18632
35 : gene_429
36 : gene_1516
37 : gene_19373
38 : gene_7343
39 : gene_8099
40 : gene_18339
41 : gene_3645
42 : gene_16372
43 : gene_742
44 : gene_5939
45 : gene_15803
46 : gene_3458
47 : gene_16987
48 : gene_10218
49 : gene_2279
50 : gene_17512
